In [1]:
import numpy as np
import pandas as pd

def entropy(y):
    """Calculate how 'mixed' the labels are. 0 = all same, 1 = completely random."""
    values, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))

def information_gain(X, y, feature):
    """How much a feature reduces uncertainty (entropy)."""
    parent_entropy = entropy(y)
    values, counts = np.unique(X[feature], return_counts=True)
    weighted_entropy = 0
    for v, c in zip(values, counts):
        subset_y = y[X[feature] == v]
        weighted_entropy += (c / len(y)) * entropy(subset_y)
    return parent_entropy - weighted_entropy

def best_feature(X, y):
    """Find which feature gives the highest information gain."""
    gains = {feature: information_gain(X, y, feature) for feature in X.columns}
    return max(gains, key=gains.get)

def id3(X, y):
    """Build the decision tree recursively (ID3 algorithm)."""
    # If all labels are the same, return that label (leaf node)
    if len(np.unique(y)) == 1:
        return y.iloc[0]
    # If no features left, return the most common label
    if X.shape[1] == 0:
        return y.value_counts().idxmax()
    # Pick the best feature to split on
    best = best_feature(X, y)
    tree = {best: {}}
    # For each value of that feature, create a subtree
    for value in X[best].unique():
        sub_X = X[X[best] == value].drop(columns=best)
        sub_y = y[X[best] == value]
        tree[best][value] = id3(sub_X, sub_y)
    return tree

def predict(tree, sample):
    """Predict the class for a single sample (dictionary of feature values)."""
    if not isinstance(tree, dict):   # leaf node -> return the label
        return tree
    feature = next(iter(tree))       # get the feature name from the tree node
    value = sample.get(feature)      # what value does the sample have for that feature?
    if value in tree[feature]:
        return predict(tree[feature][value], sample)
    else:
        return None   # value not seen in training

# ---------------- Load data from CSV ----------------
df = pd.read_csv('tennis.csv')
X = df.drop(columns='PlayTennis')
y = df['PlayTennis']

# Build the tree
tree = id3(X, y)
print("Decision tree (dictionary form):")
print(tree)

# Test with a new sample
sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}
print("\nPrediction for sample:", predict(tree, sample))

Decision tree (dictionary form):
{'Outlook': {'Sunny': {'Temperature': {'Hot': 'No', 'Cool': {'Humidity': {'High': 'No', 'Normal': {'Wind': {'Strong': 'No'}}}}, 'Mild': {'Wind': {'Weak': {'Humidity': {'Normal': 'Yes', 'High': 'Yes'}}, 'Strong': 'No'}}}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': {'Temperature': {'Mild': 'Yes', 'Cool': 'Yes', 'Hot': {'Humidity': {'Normal': 'Yes'}}}}, 'Strong': {'Temperature': {'Cool': 'No', 'Mild': {'Humidity': {'Normal': 'No'}}, 'Hot': 'No'}}}}}}

Prediction for sample: No
